In [1]:
from decouple import config
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import pandas as pd

## Criar conexao

In [2]:
POSTGRES_NAME = config('POSTGRES_NAME')
POSTGRES_USER = config('POSTGRES_USER')
POSTGRES_PASSWORD = config('POSTGRES_PASSWORD')
POSTGRES_HOST = 'localhost'
POSTGRES_PORT = config('POSTGRES_PORT')
POSTGRES_PORT = config('POSTGRES_PORT')

       
DATABASE_URL = f"postgresql+psycopg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_NAME}"

# Criando a engine do SQLAlchemy
engine = create_engine(DATABASE_URL)

## Ingestão de dados no SQL


In [5]:
link = 'https://github.com/silva-fabiofreitas/Estatistica_python/raw/main/dados/IDH.xlsx'
df = pd.read_excel(link, sheet_name='Base')
df.columns = df.columns.str.lower() 
df.head()

,espacialidades,nd,ev,mi,ps60,ql,pib,rp,gi,p,vp,fc18,sc25,eae,t18,pda,pdba,pdcl
0,Angra dos Reis,Outros,75.75,12.97,83.19,1.426171,4.540722e+06,798.68,0.50,6.69,21.42,55.41,7.42,9.00,5.43,92.49,95.45,99.26
1,Aperibé,C.Vicioso,72.10,18.40,77.85,1.113099,4.077564e+04,516.14,0.43,9.40,29.72,51.36,5.94,8.95,11.06,96.84,99.52,97.07
2,Araruama,Outros,75.32,14.18,82.88,0.409663,5.643963e+05,680.88,0.54,11.60,32.86,55.57,10.27,8.84,7.84,96.08,94.42,95.64
3,Areal,C.Vicioso,74.35,15.00,81.22,0.326421,8.742154e+04,571.74,0.48,11.13,32.21,46.76,6.24,9.21,7.77,85.66,98.20,98.57
4,Armação dos Búzios,C.Vicioso,74.44,14.80,81.36,0.054656,5.751335e+05,851.39,0.51,3.69,17.24,58.03,11.25,9.09,4.83,83.53,96.31,98.55


In [6]:
# Carregar dados no banco
df.to_sql('idh_data', con=engine, if_exists='replace', index=False)

-1

## Iformações do banco de dados

Os dados fornecidos pelo método `db.get_table_info()` do Langchain não são suficientes para compreender as informações e, às vezes, até dificultam a compreensão. É melhor montar sua própria estrutura.

In [3]:
db = SQLDatabase(engine=engine)
print(db.dialect)
print(db.get_usable_table_names())
print(db.run("SELECT COUNT(*) FROM idh_data LIMIT 2;"))
print(db.run("SELECT * FROM idh_data LIMIT 2;"))
print(db.get_table_info(['idh_data']))

postgresql
['auth_group', 'auth_group_permissions', 'auth_permission', 'auth_user', 'auth_user_groups', 'auth_user_user_permissions', 'core_category', 'core_iris', 'core_mindmap', 'data__None', 'django_admin_log', 'django_content_type', 'django_migrations', 'django_session', 'idh_data']
[(92,)]
[('Angra dos Reis', 'Outros', 75.75, 12.97, 83.19, 1.4261707521102645, 4540721.53301557, 798.68, 0.5, 6.69, 21.42, 55.41, 7.42, 9.0, 5.43, 92.49, 95.45, 99.26), ('Aperibé', 'C.Vicioso', 72.1, 18.4, 77.85, 1.1130994852579565, 40775.6429497641, 516.14, 0.43, 9.4, 29.72, 51.36, 5.94, 8.95, 11.06, 96.84, 99.52, 97.07)]

CREATE TABLE idh_data (
	espacialidades TEXT, 
	nd TEXT, 
	ev DOUBLE PRECISION, 
	mi DOUBLE PRECISION, 
	ps60 DOUBLE PRECISION, 
	ql DOUBLE PRECISION, 
	pib DOUBLE PRECISION, 
	rp DOUBLE PRECISION, 
	gi DOUBLE PRECISION, 
	p DOUBLE PRECISION, 
	vp DOUBLE PRECISION, 
	fc18 DOUBLE PRECISION, 
	sc25 DOUBLE PRECISION, 
	eae DOUBLE PRECISION, 
	t18 DOUBLE PRECISION, 
	pda DOUBLE PRECISION

### Metadados
Explicação das variaveis

In [8]:
column_descriptions = (
    ('espacialidades', 'Name of the municipalities', 'Name of the municipalities in the state of Rio de Janeiro'),
    ('nd', 'Development level', 'Classification of the level of human development (Low, Medium, High)'),
    ('mi', 'Infant mortality (%)', 'Infant mortality rate, expressed as a percentage'),
    ('ps60', 'Probability of survival to age 60 (%)', 'Probability of a person surviving to age 60'),
    ('ev', 'Life expectancy at birth', 'Average number of years a person can expect to live at birth'),
    ('ql', 'QL - Locational Quotient', 'Measure that reflects regional inequalities'),
    ('pib', 'GDP (Thousand)', 'Municipal Gross Domestic Product, in thousands of reais'),
    ('rp', 'Per capita income', 'Average income per inhabitant in the municipality'),
    ('gi', 'Gini Index', 'Measure of income inequality, ranging from 0 (total equality) to 1 (maximum inequality)'),
    ('p', '% of poor', 'Percentage of the population considered poor'),
    ('vp', '% vulnerable to poverty', 'Percentage of the population in a situation of vulnerability to poverty'),
    ('fc18', '% of 18 years or older with complete elementary education', 'Percentage of the population aged 18 or older who completed elementary school'),
    ('sc25', '% of 25 years or older with complete higher education', 'Percentage of the population aged 25 or older who completed higher education'),
    ('eae', 'Expected years of schooling', 'Average number of years a child can expect to study during their lifetime'),
    ('t18', 'Illiteracy rate - 18 years or older', 'Percentage of the population aged 18 or older who are illiterate'),
    ('pda', '% of population in households with piped water', 'Percentage of the population living in households with access to piped water'),
    ('pdba', '% of population in households with bathroom and piped water', 'Percentage of the population living in households with a bathroom and piped water'),
    ('pdcl', '% of population in households with regular garbage collection', 'Percentage of the population living in households with regular garbage collection')
)

text_description = 'TABLE: idh_data\n\n |Variável|Descrição| \n |---|---|\n'
# Display the descriptions
for column, description, description_2  in column_descriptions:
    text_description+=f"|{column}| {description_2}| \n"
print(text_description)

TABLE: idh_data

 |Variável|Descrição| 
 |---|---|
|espacialidades| Name of the municipalities in the state of Rio de Janeiro| 
|nd| Classification of the level of human development (Low, Medium, High)| 
|mi| Infant mortality rate, expressed as a percentage| 
|ps60| Probability of a person surviving to age 60| 
|ev| Average number of years a person can expect to live at birth| 
|ql| Measure that reflects regional inequalities| 
|pib| Municipal Gross Domestic Product, in thousands of reais| 
|rp| Average income per inhabitant in the municipality| 
|gi| Measure of income inequality, ranging from 0 (total equality) to 1 (maximum inequality)| 
|p| Percentage of the population considered poor| 
|vp| Percentage of the population in a situation of vulnerability to poverty| 
|fc18| Percentage of the population aged 18 or older who completed elementary school| 
|sc25| Percentage of the population aged 25 or older who completed higher education| 
|eae| Average number of years a child can expect 

## LLM

In [9]:
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=512,  # Limita o tamanho da resposta (evita SQL truncado)
    model_kwargs={
        'frequency_penalty': 0,
        'presence_penalty': 0,
        'top_p': 0.1 # Foca nos tokens mais relevantes
    }
)

## Prompt

In [10]:
from langchain import hub
from langchain_core.prompts import PromptTemplate

query_prompt_template = hub.pull("langchain-ai/sql-query-system-prompt")

assert len(query_prompt_template.messages) == 1
query_prompt_template.messages[0].pretty_print()

/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


================================ System Message ================================

Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of examples they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table, only ask for a the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.

Only use the following tables:
{table_info}

Question: {input}


In [11]:
template_sql = """Given an input question, create a syntactically correct {dialect} query to run to help find the answer. Unless the user specifies in his question a specific number of examples they wish to obtain, always limit your query to at most {top_k} results. You can order the results by a relevant column to return the most interesting examples in the database.

Never query for all the columns from a specific table; only ask for the few relevant columns given the question.

Pay attention to use only the column names that you can see in the schema description. Be careful not to query for columns that do not exist. Also, pay attention to which column is in which table.

use only the following tables:
{table_info}

Question: {input}
"""

query_prompt_template = PromptTemplate.from_template(
    template_sql,
)

# Gerar query

In [12]:
from typing_extensions import Annotated, TypedDict


from pydantic import BaseModel, Field

class QueryOutput(BaseModel):
    """Generated SQL query."""
    query: Annotated[str, Field(..., description="Syntactically valid SQL query.")]


def write_query(state: dict):
    """Generate SQL query to fetch information."""
    prompt = query_prompt_template.invoke(
        {
            "dialect": db.dialect,
            "top_k": 10,
            "table_info": text_description,
            "input": state["question"],
        }
    )
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

query = [] 
query.extend([
    write_query({"question": "Quantidade de municipios por ND"}),
    write_query({"question": "Qual o nivel de desenvolvimento dos municipios com maior mortalidade infantil"}),
    write_query({"question": "Calcule os quartis do indicador esperança de vida ao nascer, use percentis 0.25, 0.5 e 0.75"}),
    write_query({"question": "Qual a média de mortalidade infantil dos municipios com gini abaixo de 0.4"}),
    write_query({"question": "Qual a média de mortalidade infantil dos municipios com gini acima de 0.4"}),
    write_query({"question": "Qual a correlacao entre mortalidade infantil e pobreza"}),
])
query

/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(
/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(
/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` 

[{'query': 'SELECT nd, COUNT(espacialidades) AS quantidade_municipios \nFROM idh_data \nGROUP BY nd \nORDER BY quantidade_municipios DESC \nLIMIT 10;'},
 {'query': 'SELECT espacialidades, nd, mi \nFROM idh_data \nORDER BY mi DESC \nLIMIT 10;'},
 {'query': 'SELECT percentile_cont(0.25) WITHIN GROUP (ORDER BY ev) AS q1,\n       percentile_cont(0.5) WITHIN GROUP (ORDER BY ev) AS q2,\n       percentile_cont(0.75) WITHIN GROUP (ORDER BY ev) AS q3\nFROM idh_data;'},
 {'query': 'SELECT AVG(mi) AS media_mortalidade_infantil\nFROM idh_data\nWHERE gi < 0.4\nLIMIT 10;'},
 {'query': 'SELECT espacialidades, mi \nFROM idh_data \nWHERE gi > 0.4 \nORDER BY mi \nLIMIT 10;'},
 {'query': 'SELECT mi, p FROM idh_data ORDER BY mi DESC LIMIT 10;'}]

In [13]:
query_result = db.run(query[0]['query'])
query_result

"[('C.Vicioso', 65), ('Outros', 18), ('T.Desenvolvimento', 5), ('C.Virtuoso', 3), ('T.Crescimento', 1)]"

In [14]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

execute_query_tool = QuerySQLDataBaseTool(db=db)

execute_query_tool.invoke('SELECT nda, COUNT(espacialidades) AS quantidade_municipios \nFROM idh_data \nGROUP BY nd \nORDER BY quantidade_municipios DESC \nLIMIT 10;')

'Error: (psycopg.errors.UndefinedColumn) column "nda" does not exist\nLINE 1: SELECT nda, COUNT(espacialidades) AS quantidade_municipios \n               ^\nHINT:  Perhaps you meant to reference the column "idh_data.nd" or the column "idh_data.pda".\n[SQL: SELECT nda, COUNT(espacialidades) AS quantidade_municipios \nFROM idh_data \nGROUP BY nd \nORDER BY quantidade_municipios DESC \nLIMIT 10;]\n(Background on this error at: https://sqlalche.me/e/20/f405)'

## Chain gerar grafico

In [15]:
template = """Com base na amostra de dados fornecida {data}, analise as variáveis e sugira o tipo de gráfico que melhor representa os resultados. Considere os seguintes pontos:

1. **Tipo de Dados:** Identifique se os dados são categóricos, numéricos, temporais ou relacionais.
2. **Objetivo da Visualização:** Determine se o objetivo é comparar valores, mostrar distribuições, evidenciar tendências, ou destacar relações entre variáveis.
3. **Sugestão de Gráfico:** Recomende o gráfico mais adequado (ex: gráfico de barras, linhas, dispersão, pizza, histograma, boxplot, etc.) com base na natureza dos dados e no objetivo.
4. **Consideração do Usuário:** Se o usuário especificar um tipo de gráfico na pergunta {question}, priorize a sugestão do usuário, mas avalie se é adequado para os dados fornecidos. Caso não seja, explique brevemente por que outro gráfico seria mais eficaz.

"""

prompt_chart_type = PromptTemplate.from_template(template)

In [16]:
class ChartSuggestion(BaseModel):
    """Sugest chart type."""
    chart_type: list[str] = Field(..., description="Type of chart to suggest.")


In [17]:
llm_with_struct_chart = llm.with_structured_output(ChartSuggestion)

/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(
/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(


**chain**

In [18]:
chain_chart_type = prompt_chart_type | llm_with_struct_chart

In [34]:
res = chain_chart_type.invoke(
    {'data':query_result ,'question': 'Quantidade de municipios por ND'}
)
res['chart_type']

['barras', 'pizza']

In [20]:
res['chart_type'][0]

'barras'

## Chain para gerar a estrutura do grafico

In [ ]:
template = """"Voce e um especialista na biblioteca javascript echarts.
Com base no tipo de grafico fornecido {chart_type}, crie um grafico que melhor represente os resultados.
Resultados: {data}

Reposta deve ser um options do echarts usar formato json.

"""

prompt_chart_desing = PromptTemplate.from_template(template)

In [52]:
class ChartOptions(BaseModel):
    """Chart options, json."""
    options: Annotated[str, Field(..., description="Json Echarts options.")] 

In [53]:
llm_with_struct_echart = llm.with_structured_output(ChartOptions)

/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(
/home/fabiofreitas/.cache/pypoetry/virtualenvs/ia-iU1irCBy-py3.12/lib/python3.12/site-packages/pydantic/_internal/_model_construction.py:273: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use `model_fields` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.7/migration/
  warnings.warn(


In [54]:
chain_chart_desing = prompt_chart_desing | llm_with_struct_echart

In [55]:
chain_chart_desing.invoke(
    {'chart_type': res['chart_type'][0], 'data': query_result}
)

{'options': "{  \n  title: {  \n    text: 'Resultados por Categoria',  \n    subtext: 'Distribuição dos resultados',  \n    left: 'center'  \n  },  \n  tooltip: {  \n    trigger: 'item'  \n  },  \n  xAxis: {  \n    type: 'category',  \n    data: ['C.Vicioso', 'Outros', 'T.Desenvolvimento', 'C.Virtuoso', 'T.Crescimento'],  \n    name: 'Categorias'  \n  },  \n  yAxis: {  \n    type: 'value',  \n    name: 'Resultados'  \n  },  \n  series: [{  \n    name: 'Resultados',  \n    type: 'bar',  \n    data: [65, 18, 5, 3, 1],  \n    emphasis: {  \n      focus: 'series'  \n    },  \n    itemStyle: {  \n      color: '#4CAF50'  \n    }  \n  }]  \n}"}

In [43]:
import pprint

pprint.pprint(res)

{'chart_type': ['barras', 'pizza']}


In [ ]:
POSTGRES_NAME = config('POSTGRES_NAME')
POSTGRES_USER = config('POSTGRES_USER')
POSTGRES_PASSWORD = config('POSTGRES_PASSWORD')
POSTGRES_HOST = 'localhost'
POSTGRES_PORT = config('POSTGRES_PORT')
POSTGRES_PORT = config('POSTGRES_PORT')

       
DATABASE_URL = f"postgresql+psycopg://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_NAME}"

# Criando a engine do SQLAlchemy
engine = create_engine(DATABASE_URL)
db = SQLDatabase(engine=engine)
print(db.get_usable_table_names())
# print(db.run("SELECT COUNT(*) FROM idh_data LIMIT 2;"))
# print(db.run("SELECT * FROM idh_data LIMIT 2;"))
print(db.get_table_info(['data__table', 'idh_data']))
db.run('SELECT "ND" FROM "data__table";')



['auth_group', 'auth_group_permissions', 'auth_permission', 'auth_user', 'auth_user_groups', 'auth_user_user_permissions', 'core_category', 'core_iris', 'core_mindmap', 'data__table', 'django_admin_log', 'django_content_type', 'django_migrations', 'django_session', 'idh_data']

CREATE TABLE data__table (
	"Espacialidades" TEXT, 
	"ND" TEXT, 
	"EV" TEXT, 
	"Mi" TEXT, 
	"PS60" DOUBLE PRECISION, 
	"QL" DOUBLE PRECISION, 
	"PIB" DOUBLE PRECISION, 
	"RP" TEXT, 
	"Gi" DOUBLE PRECISION, 
	"P" TEXT, 
	"VP" DOUBLE PRECISION, 
	"FC18" DOUBLE PRECISION, 
	"SC25" TEXT, 
	"EAE" TEXT, 
	"T18" TEXT, 
	"PDA" DOUBLE PRECISION, 
	"PDBA" DOUBLE PRECISION, 
	"PDCL" TEXT
)

/*
3 rows from data__table table:
Espacialidades	ND	EV	Mi	PS60	QL	PIB	RP	Gi	P	VP	FC18	SC25	EAE	T18	PDA	PDBA	PDCL
Angra dos Reis	Outros	75.75	12.97	83.19	1.42617075211026	4540721.53301557	798.68	0.5	6.69	21.42	55.41	7.42	9	5.43	92.49	95.45	99.26
Aperibé	C.Vicioso	72.1	18.4	77.85	1.11309948525796	40775.6429497641	516.14	0.43	9.4	29.72	51.

"[('Outros',), ('C.Vicioso',), ('Outros',), ('C.Vicioso',), ('C.Vicioso',), ('Outros',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('Outros',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('Outros',), ('C.Vicioso',), ('T.Desenvolvimento',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('Outros',), ('Outros',), ('C.Vicioso',), ('C.Vicioso',), ('T.Crescimento',), ('C.Vicioso',), ('C.Vicioso',), ('T.Desenvolvimento',), ('T.Desenvolvimento',), ('Outros',), ('T.Desenvolvimento',), ('Outros',), ('C.Vicioso',), ('Outros',), ('Outros',), ('C.Virtuoso',), ('Outros',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('Outros',), ('C.Vicioso',), ('Outros',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('C.Vicioso',), ('T.Desenvo

In [6]:
nome = f'data_{None}'
nome

# db.run('DROP TABLE "data__None";')
# db.run('DROP TABLE "data__teste";')

db.run('PRAGMA table_info(idh_data);')

ProgrammingError: (psycopg.errors.SyntaxError) syntax error at or near "PRAGMA"
LINE 1: PRAGMA table_info(idh_data);
        ^
[SQL: PRAGMA table_info(idh_data);]
(Background on this error at: https://sqlalche.me/e/20/f405)